In [1]:
import importlib.util, subprocess, sys

_pkgs = ["langgraph", "langchain_community", "langchain_openai", "langsmith", "langgraph-supervisor"]
_missing = [p for p in _pkgs if importlib.util.find_spec(p.replace("-", "_").split("[")[0]) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _pkgs)

In [2]:
# Environment Variable Initialization

import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from the notebook's working directory

def _set_if_undefined(var_name: str):
    value = os.environ.get(var_name, "").strip()
    if value:
        masked = value[:6] + "*" * (min(20, len(value) - 6))
        print(f"  ✅ {var_name}: {masked}")
    else:
        print(f"  ❌ {var_name}: not set")

# ---- Environment Variables Required ----

print("Checking environment variables...")
_set_if_undefined("OPENAI_API_KEY")
_set_if_undefined("LANGSMITH_TRACING")
_set_if_undefined("LANGSMITH_API_KEY") # https://docs.langchain.com/langsmith/observability
_set_if_undefined("MODEL")
_set_if_undefined("OPENWEATHER_API_KEY") # https://openweathermap.org/api
print("Done.")

Checking environment variables...
  ✅ OPENAI_API_KEY: sk-pro********************
  ✅ LANGSMITH_TRACING: true
  ✅ LANGSMITH_API_KEY: lsv2_p********************
  ✅ MODEL: gpt-4o*****
  ✅ OPENWEATHER_API_KEY: 8dd1b8********************
Done.


In [3]:
# Multi-Agent Orchestration with LangGraph:
#- Supervisor agent coordinates between specialized workers.
#- Workers: weather reporting agent, dressing planner agent.
#- State management and dynamic worker routing based on conversation flow.

# ---- Imports ----

import os, warnings, logging
from typing import Literal, Annotated
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.types import Command
from langgraph.prebuilt import create_react_agent
from langgraph.warnings import LangGraphDeprecatedSinceV10

# Silence the create_react_agent V1.0 deprecation warning so demo output is clean.
warnings.filterwarnings("ignore", category=LangGraphDeprecatedSinceV10)
# Silence LangGraph's "wrote to unknown channel ... ignoring it" logger message.
logging.getLogger("langgraph").setLevel(logging.ERROR)

# ---- LLM Setup ----

# Load the default model from environment variables
default_model = os.environ["MODEL"]

# Initialize the LLM (Large Language Model) interface
llm = ChatOpenAI(model=default_model)

# ---- Supervisor Setup ----

# Define agent members (names must match the graph nodes AND the prompt exactly).
members = ["weather_reporting", "dressing_planner"]

# Supervisor options include all members + END signal
options = members + [END]

# System prompt guiding the supervisor's behavior
system_prompt = (
    "# Role and Objective"
    "You are a Supervisor Agent tasked with managing a conversation between two specialized workers: `weather_reporting` and `dressing_planner`."
    "Your goal is to orchestrate their actions to resolve the user's request efficiently."
    "# Instructions"
    f"- Persist through multiple steps until the task is fully complete. Only output {END} when finished."
    "- Always select the next worker based on context."
    "- Think step-by-step before choosing a worker and after receiving results."
    "# Reasoning Steps"
    "1. Analyze current state."
    "2. Plan the next best action."
    "3. Reflect after worker output."
    "4. Repeat until completion."
    "5. End cleanly with {END}."
    "# Tool/Worker Use"
    "- `weather_reporting`: gather or analyze weather."
    "- `dressing_planner`: suggest clothing based on weather."
)

class Router(TypedDict):
    """Defines the next worker or signals END."""
    next: Literal[*options]

class State(MessagesState):
    """Extended state tracking next agent selection."""
    next: str

# ---- Node Definitions ----

def supervisor_node(state: State) -> Command[Literal[*members, END]]:
    """Supervisor agent logic to pick next worker."""
    messages = [{"role": "system", "content": system_prompt}] + state["messages"]
    response = llm.with_structured_output(Router).invoke(messages)
    goto = response["next"]
    return Command(goto=goto, update={"next": goto})
  

In [4]:
# ---- Worker Agents ----

import requests

@tool
def weather_reporting_tool(city: Annotated[str, "name of the city"]):
    """Tool to fetch current weather data for a given city from OpenWeatherMap."""
    resp = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={
            "q": city,
            "appid": os.environ["OPENWEATHER_API_KEY"],
            "units": "metric",  # temperatures in Celsius
        },
        timeout=10,
    )
    resp.raise_for_status()
    data = resp.json()
    return {
        "weather": {
            "name": data["name"],
            "main": data["weather"][0]["main"],
            "description": data["weather"][0]["description"],
            "units": "metric (Celsius)",
            **data["main"],   # temp, feels_like, temp_min, temp_max, pressure, humidity
            "wind_speed": data.get("wind", {}).get("speed"),
        }
    }

# Weather Reporting Agent
weather_reporting_agent = create_react_agent(
    llm,
    tools=[weather_reporting_tool],
    name="weather_reporting_agent",
    prompt=(
        "You are a weather reporter. Report current weather for the provided city. "
        "You may use tools. Temperatures are in Celsius. Do not suggest what to wear."
    )
)

def weather_reporting_node(state: State) -> Command[Literal["supervisor"]]:
    """Node to execute weather reporting."""
    result = weather_reporting_agent.invoke(state)
    return Command(
        update={
            "messages": [
                HumanMessage(content=result["messages"][-1].content, name="weather_reporting")
            ]
        },
        goto="supervisor",
    )

# Dressing Planner Agent
dressing_planner_agent = create_react_agent(
    llm,
    tools=[],
    name="dressing_planner_agent",
    prompt=(
        "You suggest dressing options based on the current weather. "
        "Prioritize 'feels like' temperature and consider wind conditions."
    )
)

def dressing_planner_node(state: State) -> Command[Literal["supervisor"]]:
    """Node to execute dressing planning based on weather."""
    result = dressing_planner_agent.invoke(state)
    return Command(
        update={
            "messages": [
                HumanMessage(content=result["messages"][-1].content, name="dressing_planner")
            ]
        },
        goto="supervisor",
    )

In [5]:
# ---- Graph Assembly ----

# Create the state graph
builder = StateGraph(State)
builder.add_edge(START, "supervisor")
builder.add_node("supervisor", supervisor_node)
builder.add_node("weather_reporting", weather_reporting_node)
builder.add_node("dressing_planner", dressing_planner_node)

# Compile the graph
graph = builder.compile()

In [6]:
# ---- Simulation ----

for s in graph.stream(
    {"messages": [("user", "Stockholm")]}, debug=True):
    print(s)
    print("============================")

[values] {'messages': [HumanMessage(content='Stockholm', additional_kwargs={}, response_metadata={}, id='772987b7-12e3-4943-935c-3ecfbf9d9c59')]}
[updates] {'supervisor': {'next': 'weather_reporting'}}
{'supervisor': {'next': 'weather_reporting'}}
[values] {'messages': [HumanMessage(content='Stockholm', additional_kwargs={}, response_metadata={}, id='772987b7-12e3-4943-935c-3ecfbf9d9c59')], 'next': 'weather_reporting'}
[updates] {'weather_reporting': {'messages': [HumanMessage(content='The current weather in Stockholm is characterized by overcast clouds. The temperature is approximately 16.2°C, with a slight breeze at a wind speed of 6.7 m/s. Humidity levels are at 84%, and the atmospheric pressure is 999 hPa.', additional_kwargs={}, response_metadata={}, name='weather_reporting', id='568fc6ea-d98e-43e5-8af6-81ace173b434')]}}
{'weather_reporting': {'messages': [HumanMessage(content='The current weather in Stockholm is characterized by overcast clouds. The temperature is approximately 1